
# **Training: CNN-Based Kitchen Cleanliness Classification**

This notebook implements a Convolutional Neural Network (CNN) pipeline for binary image classification focused on kitchen cleanliness detection.

The objective is to classify images into two categories:

- `clean`
- `dirty`

The workflow includes:
- dataset loading
- CNN training
- model evaluation
- metric extraction and JSON export

The datasets are already preprocessed and divided into:

- training set
- validation set
- test set

with the following directory structure:

```text
dataset/
    train/
        clean/
        dirty/

    val/
        clean/
        dirty/

    test/
        clean/
        dirty/
````




# **Libraries used**

In [1]:
from pathlib import Path
import tensorflow as tf
import numpy as np
import pandas as pd
import cv2
import os
import tensorflow as tf
import json
import time

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)


import matplotlib.pyplot as plt
import seaborn as sns

# **Auxiliar functions**

## **1) Load Dataset**

In [2]:
def load_dataset_for_cnn(
    dataset_dir,
    image_size=(224, 224),
    batch_size=32,
    shuffle=True
):
    """
    Returns:
        train_ds, val_ds, test_ds, y_train, y_val, y_test, class_names
    where:
        - train_ds, val_ds, test_ds: tf.data.Dataset objects for training, validation, and testing.
        - y_train, y_val, y_test: Numpy arrays containing the labels for the training, validation, and testing datasets.
        - class_names: List of class names corresponding to the labels.
    """

    dataset_dir = Path(dataset_dir)

    train_dir = dataset_dir / "train"
    val_dir = dataset_dir / "val"
    test_dir = dataset_dir / "test"

    
    # LOAD DATASETS
    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir,
        labels="inferred",
        label_mode="binary",
        image_size=image_size,
        batch_size=batch_size,
        shuffle=shuffle
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        val_dir,
        labels="inferred",
        label_mode="binary",
        image_size=image_size,
        batch_size=batch_size,
        shuffle=False
    )

    test_ds = tf.keras.utils.image_dataset_from_directory(
        test_dir,
        labels="inferred",
        label_mode="binary",
        image_size=image_size,
        batch_size=batch_size,
        shuffle=False
    )

    # SAVE CLASS NAMES
    class_names = train_ds.class_names

    # NORMALIZATION
    normalization_layer = tf.keras.layers.Rescaling(1.0 / 255)

    train_ds = train_ds.map(
        lambda x, y: (normalization_layer(x), y)
    )

    val_ds = val_ds.map(
        lambda x, y: (normalization_layer(x), y)
    )

    test_ds = test_ds.map(
        lambda x, y: (normalization_layer(x), y)
    )

    # EXTRACT LABELS
    y_train = np.concatenate([
        y.numpy() for _, y in train_ds
    ])

    y_val = np.concatenate([
        y.numpy() for _, y in val_ds
    ])

    y_test = np.concatenate([
        y.numpy() for _, y in test_ds
    ])

    # Flatten labels
    y_train = y_train.flatten()
    y_val = y_val.flatten()
    y_test = y_test.flatten()

  
    # PERFORMANCE
    AUTOTUNE = tf.data.AUTOTUNE

    train_ds = train_ds.prefetch(AUTOTUNE)
    val_ds = val_ds.prefetch(AUTOTUNE)
    test_ds = test_ds.prefetch(AUTOTUNE)

    return (
        train_ds,
        val_ds,
        test_ds,
        y_train,
        y_val,
        y_test,
        class_names
    )

## **2) CNN pipeline** 

The implemented model is a sequential Convolutional Neural Network (CNN) designed for binary image classification (`clean` vs `dirty`).


# Architecture Table

| Layer Type | Parameters | Output Purpose |
|---|---|---|
| Input Layer | `(224, 224, 3)` | Receives RGB images resized to 224×224 |
| Conv2D | 32 filters, 3×3 kernel, ReLU | Extracts low-level visual features |
| MaxPooling2D | 2×2 pool size | Reduces spatial dimensions |
| Conv2D | 64 filters, 3×3 kernel, ReLU | Learns intermediate visual patterns |
| MaxPooling2D | 2×2 pool size | Downsampling |
| Conv2D | 128 filters, 3×3 kernel, ReLU | Learns higher-level semantic features |
| MaxPooling2D | 2×2 pool size | Further dimensionality reduction |
| Flatten | — | Converts feature maps into a 1D vector |
| Dense | 128 neurons, ReLU | Fully connected feature learning |
| Dropout | 0.5 | Reduces overfitting |
| Output Dense | 1 neuron, Sigmoid | Produces binary classification probability |



In [3]:
def build_and_train_cnn(
    train_ds,
    val_ds,
    input_shape=(224, 224, 3),
    num_classes=1,
    epochs=10,
    learning_rate=0.001
):
    """
    Builds and trains a simple CNN for binary classification.

    Returns:
        model
        history
    """

    model = tf.keras.Sequential([

        # Conv Block 1
        tf.keras.layers.Conv2D(
            32,
            (3, 3),
            activation="relu",
            input_shape=input_shape
        ),

        tf.keras.layers.MaxPooling2D((2, 2)),

        # Conv Block 2
        tf.keras.layers.Conv2D(
            64,
            (3, 3),
            activation="relu"
        ),

        tf.keras.layers.MaxPooling2D((2, 2)),

        # Conv Block 3
        tf.keras.layers.Conv2D(
            128,
            (3, 3),
            activation="relu"
        ),

        tf.keras.layers.MaxPooling2D((2, 2)),

        # Flatten
        tf.keras.layers.Flatten(),

        # Dense Layers
        tf.keras.layers.Dense(
            128,
            activation="relu"
        ),

        tf.keras.layers.Dropout(0.5),

        # Output Layer
        # Binary classification
        tf.keras.layers.Dense(
            num_classes,
            activation="sigmoid"
        )
    ])

    # COMPILE

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),

        loss="binary_crossentropy",

        metrics=[
            "accuracy",
            tf.keras.metrics.Precision(),
            tf.keras.metrics.Recall()
        ]
    )

   

    model.summary()

    # TRAIN
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs
    )

    return model, history

## **3) Metrics**

### **1) Folders**
This function check if the paths already exist or create them.

In [4]:
def create_experiment_folders():

    models_dir = Path("models")
    results_dir = Path("results")

    models_dir.mkdir(exist_ok=True)
    results_dir.mkdir(exist_ok=True)

    return models_dir, results_dir


### **2) Save models**
This function check if the paths already exist or create them.

In [5]:
def save_model(model, model_name="cnn_model"):
    models_dir, _ = create_experiment_folders()
    model_path = models_dir / f"{model_name}.keras"
    model.save(model_path)
    print(f"\nModel saved at: {model_path}")
    return model_path


def save_metrics_json(
    metrics,
    save_path="results/metrics.json"
):
    with open(save_path, "w") as f:
        json.dump(metrics, f, indent=4)
    print(f"Metrics saved at: {save_path}")

This function print the confusion matrix in a more readable way

In [6]:
def plot_confusion_matrix_paper_style(
    y_test,
    y_pred,
    class_names,
    save_path="results/confusion_matrix.png"
):
    """
    Generates paper-style confusion matrix.
    """

    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(6, 6))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names
    )
    disp.plot(
        cmap="Blues",
        ax=ax,
        colorbar=False
    )

    plt.title("Confusion Matrix")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.tight_layout()
    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()
    print(f"Confusion matrix saved at: {save_path}")

In [7]:
def evaluate_model(
    model,
    test_ds,
    y_test,
    class_names,
    model_name="cnn_model",
    threshold=0.5
):
    """
    Evaluates CNN model.
    Saves:
        - model
        - metrics json
        - confusion matrix image
    """

    # CREATE FOLDERS
    models_dir, results_dir = create_experiment_folders()


    # INFERENCE
    start_time = time.time()
    y_probs = model.predict(test_ds)
    end_time = time.time()
    inference_time = end_time - start_time

    # PREDICTIONS
    y_probs = y_probs.flatten()
    y_pred = (y_probs >= threshold).astype(int)

    # METRICS
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_probs)
    report = classification_report(
        y_test,
        y_pred,
        output_dict=True
    )
    cm = confusion_matrix(
        y_test,
        y_pred
    )

    # RESULTS DICTIONARY
    metrics = {

        "model_name": model_name,
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1_score": float(f1),
        "auc": float(auc),
        "inference_time_seconds": float(
            inference_time
        ),
        "num_test_samples": int(
            len(y_test)
        ),
        "classification_report": report,
        "confusion_matrix": cm.tolist()
    }


    # PRINT RESULTS
    print("\n========== MODEL METRICS ==========\n")

    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1 Score  : {f1:.4f}")
    print(f"AUC       : {auc:.4f}")
    print(
        f"\nInference Time: "
        f"{inference_time:.4f} seconds"
    )
    print("\n========== CLASSIFICATION REPORT ==========\n")
    print(
        classification_report(
            y_test,
            y_pred
        )
    )

    # SAVE MODEL
    save_model(
        model=model,
        model_name=model_name
    )

    # SAVE METRICS JSON
    metrics_path = (
        results_dir /
        f"{model_name}_metrics.json"
    )

    save_metrics_json(
        metrics=metrics,
        save_path=metrics_path
    )

    # SAVE CONFUSION MATRIX
    cm_path = (
        results_dir /
        f"{model_name}_confusion_matrix.png"
    )

    plot_confusion_matrix_paper_style(
        y_test=y_test,
        y_pred=y_pred,
        class_names=class_names,
        save_path=cm_path
    )

    return metrics

In [8]:
ORIGINAL_SPLIT_PATH = "No_augmented_images_split"
AUGMENTED_SPLIT_PATH = "Augmented_images_split"

# **CNN: Original Data**

In [9]:
train_ds, val_ds, test_ds, y_train, y_val, y_test, class_names = load_dataset_for_cnn(
    dataset_dir=ORIGINAL_SPLIT_PATH,
    image_size=(224, 224),
    batch_size=32
)

Found 209 files belonging to 2 classes.


Found 45 files belonging to 2 classes.
Found 46 files belonging to 2 classes.


In [10]:
model, history = build_and_train_cnn(
    train_ds=train_ds,
    val_ds=val_ds,
    input_shape=(224, 224, 3),
    epochs=15,
    learning_rate=0.001
)

c:\Users\Ale\Downloads\DepaY-All-Files\Codigo\Cocina\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,089 (42.61 MB)

 Trainable params: 11,169,089 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 4s 298ms/step - accuracy: 0.5359 - loss: 1.0565 - precision: 0.6308 - recall: 0.6260 - val_accuracy: 0.6222 - val_loss: 0.6490 - val_precision: 0.6222 - val_recall: 1.0000
Epoch 2/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 320ms/step - accuracy: 0.6268 - loss: 0.6320 - precision: 0.6268 - recall: 1.0000 - val_accuracy: 0.6444 - val_loss: 0.6848 - val_precision: 0.9286 - val_recall: 0.4643
Epoch 3/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 314ms/step - accuracy: 0.6364 - loss: 0.6550 - precision: 0.6950 - recall: 0.7481 - val_accuracy: 0.6222 - val_loss: 0.6576 - val_precision: 0.6222 - val_recall: 1.0000
Epoch 4/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 227ms/step - accuracy: 0.6220 - loss: 0.6081 - precision: 0.6275 - recall: 0.9771 - val_accuracy: 0.6222 - val_loss: 0.5965 - val_precision: 0.6222 - val_recall: 1.0000
Epoch 5/15
7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 336ms/step - accuracy: 0.7560 - loss: 0.4816 - precision: 0.7273 - recall: 0.9771 - val_accuracy: 0.7333 - val_loss: 0.5289 - val

In [11]:
metrics = evaluate_model(
    model=model,
    test_ds=test_ds,
    y_test=y_test,
    class_names=class_names,
    model_name="cnn_no_augmented"
)

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 193ms/step

========== MODEL METRICS ==========

Accuracy  : 0.7826
Precision : 0.7568
Recall    : 0.9655
F1 Score  : 0.8485
AUC       : 0.9006

Inference Time: 0.7181 seconds

========== CLASSIFICATION REPORT ==========

              precision    recall  f1-score   support

         0.0       0.89      0.47      0.62        17
         1.0       0.76      0.97      0.85        29

    accuracy                           0.78        46
   macro avg       0.82      0.72      0.73        46
weighted avg       0.81      0.78      0.76        46


Model saved at: models\cnn_no_augmented.keras
Metrics saved at: results\cnn_no_augmented_metrics.json
Confusion matrix saved at: results\cnn_no_augmented_confusion_matrix.png


# **CNN: Augmented Data**

In [12]:
train_ds, val_ds, test_ds, y_train, y_val, y_test, class_names = load_dataset_for_cnn(
    dataset_dir=AUGMENTED_SPLIT_PATH,
    image_size=(224, 224),
    batch_size=32
)

Found 1254 files belonging to 2 classes.
Found 45 files belonging to 2 classes.
Found 46 files belonging to 2 classes.


In [13]:
model, history = build_and_train_cnn(
    train_ds=train_ds,
    val_ds=val_ds,
    input_shape=(224, 224, 3),
    epochs=15,
    learning_rate=0.001
)

c:\Users\Ale\Downloads\DepaY-All-Files\Codigo\Cocina\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,089 (42.61 MB)

 Trainable params: 11,169,089 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 12s 259ms/step - accuracy: 0.6507 - loss: 0.6488 - precision_1: 0.6737 - recall_1: 0.8588 - val_accuracy: 0.6667 - val_loss: 0.7395 - val_precision_1: 0.6512 - val_recall_1: 1.0000
Epoch 2/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 11s 249ms/step - accuracy: 0.7656 - loss: 0.4948 - precision_1: 0.7943 - recall_1: 0.8448 - val_accuracy: 0.7111 - val_loss: 0.5299 - val_precision_1: 0.8947 - val_recall_1: 0.6071
Epoch 3/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 11s 265ms/step - accuracy: 0.8405 - loss: 0.3714 - precision_1: 0.8737 - recall_1: 0.8715 - val_accuracy: 0.7333 - val_loss: 0.4701 - val_precision_1: 0.8077 - val_recall_1: 0.7500
Epoch 4/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 10s 239ms/step - accuracy: 0.8955 - loss: 0.2551 - precision_1: 0.9109 - recall_1: 0.9237 - val_accuracy: 0.7556 - val_loss: 0.5547 - val_precision_1: 0.9474 - val_recall_1: 0.6429
Epoch 5/15
40/40 ━━━━━━━━━━━━━━━━━━━━ 11s 267ms/step - accuracy: 0.9402 - loss: 0.1512 - precision_1: 0.9540 - recall_1: 0.9

In [14]:
metrics = evaluate_model(
    model=model,
    test_ds=test_ds,
    y_test=y_test,
    class_names=class_names,
    model_name="cnn_augmented"
)

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step

========== MODEL METRICS ==========

Accuracy  : 0.9348
Precision : 0.9643
Recall    : 0.9310
F1 Score  : 0.9474
AUC       : 0.9899

Inference Time: 0.5248 seconds

========== CLASSIFICATION REPORT ==========

              precision    recall  f1-score   support

         0.0       0.89      0.94      0.91        17
         1.0       0.96      0.93      0.95        29

    accuracy                           0.93        46
   macro avg       0.93      0.94      0.93        46
weighted avg       0.94      0.93      0.94        46


Model saved at: models\cnn_augmented.keras
Metrics saved at: results\cnn_augmented_metrics.json
Confusion matrix saved at: results\cnn_augmented_confusion_matrix.png
